# L5 · Lab D — The Value Head, Dissected

**RL for LLMs / RLHF — Vizuara AI Labs · Dr. Rajat Dandekar**

The lecture's most surprising point: **the value function is just a second head on the same transformer.** One shared trunk, two heads — the **policy head** (actor, next-token distribution) and the **value head** (critic, one scalar per position estimating the eventual reward).

You will:
1. Look inside the value head — confirm it's a `Linear(768 → 1)`.
2. Train it to **predict the return** at every token (plain regression).
3. Show two things the lecture claims: the critic **learns** (MSE crashes, V↔return correlation is high), and it is **more confident at the end than the start** (last-token error ≪ first-token error).

> Runtime: **GPU**. ~5 minutes.

In [ ]:
!pip -q install "transformers==4.45.2" "trl==0.11.4" "datasets==3.0.1" >/dev/null

In [ ]:
# ✅ PROVIDED — models: policy (with a VALUE HEAD) + frozen reference + reward model
import os, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from trl import AutoModelForCausalLMWithValueHead
from datasets import load_dataset
device = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained("gpt2"); tok.pad_token = tok.eos_token; tok.truncation_side = "left"
policy = AutoModelForCausalLMWithValueHead.from_pretrained("gpt2").to(device)   # actor + value head
ref    = AutoModelForCausalLMWithValueHead.from_pretrained("gpt2").to(device).eval()
torch.manual_seed(0)
with torch.no_grad():
    for p in policy.parameters():
        if p.dim() >= 2: p.add_(0.1 * p.std() * torch.randn_like(p))   # a drifted policy
policy.eval()

# reward model (load Lab A's ./rm, else quick-train)
rm_tok = AutoTokenizer.from_pretrained("gpt2"); rm_tok.pad_token = rm_tok.eos_token; rm_tok.truncation_side = "left"
if os.path.isdir("rm"):
    rm = AutoModelForSequenceClassification.from_pretrained("rm").to(device).eval()
else:
    rm = AutoModelForSequenceClassification.from_pretrained("gpt2", num_labels=1)
    rm.config.pad_token_id = rm_tok.eos_token_id; rm.to(device)
    _d = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=0).select(range(4000))
    def _sc(t):
        e = rm_tok(t, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        return rm(**e).logits.squeeze(-1)
    _o = torch.optim.AdamW(rm.parameters(), lr=2e-5)
    for i in range(0, 4000, 8):
        rc, rr = _sc(_d["chosen"][i:i+8]), _sc(_d["rejected"][i:i+8])
        L = -F.logsigmoid(rc-rr).mean(); _o.zero_grad(); L.backward(); _o.step()
    rm.eval()
@torch.no_grad()
def rm_score(text):
    e = rm_tok(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    return rm(**e).logits.squeeze().item()

## 1 · The value head is a scalar head on the transformer

In [ ]:
# ✅ PROVIDED — inspect the value head
vh = policy.v_head
print("value head type:", type(vh).__name__)
print("maps hidden state of size", vh.summary.in_features, "->", vh.summary.out_features, "scalar")
# one forward pass returns BOTH the token logits AND a value per position:
ids = tok("Hello there", return_tensors="pt").input_ids.to(device)
logits, _, values = policy(ids)
print("logits per token:", tuple(logits.shape), " | value per token:", tuple(values.shape))

## 2 · Collect rollouts and their returns

We sample responses, then build the per-token **return** `G_t` (reward-to-go = terminal reward-model score plus the small per-token KL tolls). The value head's job is to predict these.

In [ ]:
# ✅ PROVIDED — rollouts -> per-token returns G
BETA = 0.2
data = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=0).select(range(300))
seqs, rets = [], []
for i in range(80):
    cut = data["chosen"][i].rfind("Assistant:"); prompt = data["chosen"][i][:cut+len("Assistant:")]
    pids = tok(prompt, return_tensors="pt", truncation=True, max_length=96).input_ids.to(device)
    with torch.no_grad():
        gen = policy.generate(pids, max_new_tokens=20, do_sample=True, top_p=0.9, top_k=0, pad_token_id=tok.eos_token_id)
    n = gen.shape[1] - pids.shape[1]
    if n < 2: continue
    with torch.no_grad():
        lp = F.log_softmax(policy(gen)[0][:, :-1], -1).gather(-1, gen[:, 1:].unsqueeze(-1)).squeeze(-1)[0][-n:]
        lr_= F.log_softmax(ref(gen)[0][:, :-1], -1).gather(-1, gen[:, 1:].unsqueeze(-1)).squeeze(-1)[0][-n:]
        R = -BETA * (lp - lr_); R[-1] += rm_score(tok.decode(gen[0]))
    G = torch.flip(torch.cumsum(torch.flip(R, [0]), 0), [0])
    seqs.append(gen); rets.append(G)
allG = torch.cat(rets)
print("collected", len(seqs), "rollouts;  return mean", round(allG.mean().item(),3), " var", round(allG.var().item(),3))

## 3 · 📝 TODO — read the value head, and train it to predict the return

Two method-defining lines: get the per-token value from the value head, and the value loss (mean-squared error to the return).

In [ ]:
def value_of(gen, n):
    # 📝 TODO: return the value head's per-token values for the LAST n positions of `gen`.
    # The wrapper's forward returns (logits, loss, value); value is [batch, seq].
    raise NotImplementedError("return policy(gen)[-1][0][-n:]")

def value_loss(v, G):
    # 📝 TODO: regression loss making the value v predict the return G.
    raise NotImplementedError("return F.mse_loss(v, G)")

In [ ]:
# ✅ PROVIDED — MSE before, then train ONLY the value head, then MSE/corr after
with torch.no_grad():
    mse_before = F.mse_loss(torch.cat([value_of(g, G.shape[0]) for g, G in zip(seqs, rets)]), allG).item()

for p in policy.parameters(): p.requires_grad_(False)
for p in policy.v_head.parameters(): p.requires_grad_(True)
opt = torch.optim.Adam(policy.v_head.parameters(), lr=5e-3)
for step in range(200):
    opt.zero_grad(); tot = 0.0
    for g, G in zip(seqs, rets):
        loss = value_loss(value_of(g, G.shape[0]), G); loss.backward(); tot += loss.item()
    opt.step()

with torch.no_grad():
    Vt, first_se, last_se = [], [], []
    for g, G in zip(seqs, rets):
        v = value_of(g, G.shape[0]); Vt.append(v)
        first_se.append((v[0]-G[0]).pow(2).item()); last_se.append((v[-1]-G[-1]).pow(2).item())
    Vall = torch.cat(Vt); mse_after = F.mse_loss(Vall, allG).item()
    vm, gm = Vall-Vall.mean(), allG-allG.mean()
    corr = float((vm*gm).sum()/(vm.norm()*gm.norm()+1e-8))
print(f"value MSE   before {mse_before:.2f}  ->  after {mse_after:.2f}")
print(f"corr(V, return) after training: {corr:.3f}")
print(f"MSE at FIRST token {sum(first_se)/len(first_se):.2f}   vs   LAST token {sum(last_se)/len(last_se):.2f}")

## 4 · The critic sharpens as the answer completes

In [ ]:
# ✅ PROVIDED — predicted value vs actual return
import matplotlib.pyplot as plt
plt.figure(figsize=(5,5))
plt.scatter(allG.cpu(), Vall.cpu(), s=8, alpha=0.4, c="#3D5A4A")
lo, hi = float(allG.min()), float(allG.max())
plt.plot([lo,hi],[lo,hi],"--",c="gray"); plt.xlabel("actual return G"); plt.ylabel("value head V")
plt.title(f"The critic learned to predict the return (r={corr:.2f})"); plt.show()

## 5 · Questions to answer

1. In one sentence, in what concrete sense is the value function "a head on the transformer"?
2. The value head is **more accurate at the last token than the first**. Why can't it predict the final reward well at the *start* of a response? (This is exactly why the lecture said *V is uncertain early, sharp late*.)
3. The advantage is `A = G − V`. Using your trained critic, why does subtracting V give a **lower-variance** training signal than using the raw return G? (Recall the baseline argument from Lecture 3.)
4. In PPO the value head and the policy head are trained **together** every step. Why must the critic be retrained continuously as the policy changes?
